# Medical + Social Heat Outreach Assessment (v0.3)

The active model is `heat_outreach_model_definition_v0_3.json`. It scores four domains:

- clinical susceptibility: `0–2`;
- medication considerations: `0–2`;
- cooling access: `0–2`;
- support and assistance: `0–2`.

Together they form the outreach-vulnerability score `V = 0–8`. Expected heat exposure, cooling breaks, and strenuous activity are **not scored**. They may optionally tailor an action after priority is selected, but they cannot change the score, score range, status, or outreach priority.

The environmental input comes from the official NWS National Digital Forecast Database XML service. The service accepts a ZIP code and returns the NWS Experimental HeatRisk category for each local forecast period. HeatRisk `H = 0–4` remains separate from patient vulnerability `V = 0–8`; never add H and V together.

Use one function:

```python
result = assess_patient_for_zip(patient, zip_code, forecast_date)
```

The patient is a plain dictionary using the field names from the JSON model. Use `None` or omit a key when scored information is unknown. Supply any five-digit U.S. ZIP code and either a local forecast date in `YYYY-MM-DD` format or `None` for today's date at that ZIP. ZIP codes that begin with zero must be strings, such as `"02108"`. The function returns score ranges for missing scored data and never treats unknown as “no.” `assess_heat_outreach(...)` remains available for offline or synthetic HeatRisk values.

Only the ZIP code is sent to NWS; do not send patient identifiers or clinical information. This remains a non-validated preventive-outreach prototype. It does not diagnose illness, estimate individual probability, or recommend medication or fluid changes.


## v0.3 scoring policy

### Clinical susceptibility: 0–2

Add one point each for age 65+, heart failure, chronic kidney disease, diabetes, and previous heat illness, then cap the domain at 2.

### Medication considerations: 0–2

- `0`: medication review complete and no mapped heat-relevant class;
- `1`: at least one mapped heat-relevant class;
- `2`: diuretic plus ACE inhibitor, ARB, or ARNI.

### Cooling access: 0–2

- `0`: reliable home cooling;
- `1`: no reliable home cooling, but a confirmed usable alternative cooling location;
- `2`: neither reliable home cooling nor a usable alternative.

Alternative cooling is “usable” only when its opening hours, transportation, affordability, and accessibility are workable for the patient.

### Support and assistance: 0–2

- add `1` when the patient does not have reliable check-in support;
- add `1` when the patient needs help reaching cooling or carrying out the heat-safety plan.

Living alone is not itself a point; actual support is assessed.

### Not scored: heat exposure

`expected_heat_exposure`, `has_reliable_cooling_breaks`, and `has_strenuous_heat_exposure` do not affect V, its band, assessment status, or outreach priority.

The four-domain score is `V = 0–8`. The original guide's cut points are retained, with the upper band ending at the new maximum of 8:

| Official NWS HeatRisk | V 0–2 | V 3–5 | V 6–8 |
|---|---|---|---|
| 0 Little to none | Routine | Routine | Routine |
| 1 Minor | Routine | Targeted | Targeted |
| 2 Moderate | Targeted | Targeted | Priority |
| 3 Major | Targeted | Priority | Priority |
| 4 Extreme | Priority | Priority | Priority |

Unknown scored inputs produce minimum and maximum domain scores. The function evaluates every score in that range: if every completion produces the same priority, it returns that priority with `partial` status; if priority could change, it returns `needs_information` and lists the possible priorities.

These weights, bands, and matrix cells are provisional project policy, not a validated clinical risk model.


In [1]:
from datetime import date, datetime, timezone
from itertools import product
from pathlib import Path
from uuid import uuid4
import json
import re


MODEL_PATH = Path("heat_outreach_model_definition_v0_3.json")
MODEL = json.loads(MODEL_PATH.read_text(encoding="utf-8"))

if (
    MODEL.get("model_id") != "hypertension_heat_outreach"
    or MODEL.get("version") != "0.3.0"
):
    raise ValueError(
        "This notebook implements hypertension_heat_outreach version 0.3.0 only."
    )

SCORING = MODEL["scoring"]
MATRIX = SCORING["priority_matrix"]
MATRIX_COLUMNS = SCORING["matrix_column_order"]
HEAT_RELEVANT_CLASSES = set(
    next(item for item in SCORING["domains"] if item["id"] == "medication")[
        "heat_relevant_classes"
    ]
)
PRIORITY_ORDER = ["routine", "targeted", "priority"]

CLINICAL_BOOLEAN_FIELDS = [
    "has_heart_failure",
    "has_chronic_kidney_disease",
    "has_diabetes",
    "history_of_heat_illness",
]

CORE_BOOLEAN_FIELDS = [
    "has_hypertension",
    *CLINICAL_BOOLEAN_FIELDS,
    "medication_review_complete",
    "has_reliable_home_cooling",
    "can_access_cooling_location",
    "has_reliable_checkin_support",
    "needs_assistance_for_heat_protection",
    "has_prescribed_fluid_restriction",
]

# These fields may tailor REDUCE_HEAT_EXPOSURE after priority is fixed. They
# never affect score, score range, band, missing-data status, or priority.
EXPOSURE_ACTION_CONTEXT_FIELDS = [
    "expected_heat_exposure",
    "has_reliable_cooling_breaks",
    "has_strenuous_heat_exposure",
]


def _token(value):
    return re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", value.strip().lower())).strip("_")


def _symptom_route(symptoms):
    emergency = set(MODEL["symptom_policy"]["emergency_signals"])
    review = set(MODEL["symptom_policy"]["other_new_symptoms"])

    if symptoms is None or not isinstance(symptoms, list):
        return (
            "not_assessed",
            "Symptoms were not assessed. This preventive assessment does not rule out heat illness.",
            ["SYMPTOMS_NOT_ASSESSED"],
        )

    normalized = {
        _token(value)
        for value in symptoms
        if isinstance(value, str) and value.strip()
    }
    if normalized & emergency:
        return (
            "emergency",
            "Call 911 now (US). Move to a cooler place and begin cooling while help arrives. "
            "Do not give fluids to an unconscious person.",
            ["EMERGENCY_SYMPTOM_REPORTED"],
        )
    if normalized:
        code = (
            "HEAT_RELATED_SYMPTOM_REPORTED"
            if normalized & review
            else "REPORTED_SYMPTOM_REQUIRES_REVIEW"
        )
        return (
            "symptom_review",
            "Move to a cooler place and contact a clinician or urgent care for prompt assessment. "
            "Seek prompt care for vomiting or worsening symptoms. Call 911 for confusion, "
            "seizure, or loss of consciousness.",
            [code],
        )
    return "none_reported", None, []


def _clinical_score(patient):
    known = int(patient["age"] >= 65)
    unknown = 0
    for field in CLINICAL_BOOLEAN_FIELDS:
        value = patient.get(field)
        known += int(value is True)
        unknown += int(value is None)
    return {"min": min(2, known), "max": min(2, known + unknown)}


def _medication_score(patient):
    classes = patient.get("medication_classes")
    review_complete = patient.get("medication_review_complete") is True
    known_classes = {value.strip().upper() for value in (classes or [])}
    relevant = known_classes & HEAT_RELEVANT_CLASSES

    combo = "DIURETIC" in relevant and bool(relevant & {"ACEI", "ARB", "ARNI"})
    if combo:
        return {"min": 2, "max": 2}
    if relevant:
        return {"min": 1, "max": 1 if review_complete and classes is not None else 2}
    if review_complete and classes is not None:
        return {"min": 0, "max": 0}
    return {"min": 0, "max": 2}


def _possible_boolean_values(value):
    return [False, True] if value is None else [value]


def _cooling_score(patient):
    values = []
    for home, alternative in product(
        _possible_boolean_values(patient.get("has_reliable_home_cooling")),
        _possible_boolean_values(patient.get("can_access_cooling_location")),
    ):
        values.append(0 if home else (1 if alternative else 2))

    reasons = []
    if patient.get("has_reliable_home_cooling") is False:
        reasons.append("NO_RELIABLE_HOME_COOLING")
    if (
        patient.get("has_reliable_home_cooling") is not True
        and patient.get("can_access_cooling_location") is not True
    ):
        reasons.append("COOLING_ACCESS_NOT_CONFIRMED")
    return {"min": min(values), "max": max(values)}, reasons


def _support_score(patient):
    checkin = patient.get("has_reliable_checkin_support")
    assistance = patient.get("needs_assistance_for_heat_protection")
    minimum = int(checkin is False) + int(assistance is True)
    maximum = minimum + int(checkin is None) + int(assistance is None)

    reasons = []
    if checkin is False:
        reasons.append("NO_RELIABLE_CHECKIN_SUPPORT")
    if assistance is True:
        reasons.append("NEEDS_HEAT_PROTECTION_ASSISTANCE")
    return {"min": minimum, "max": maximum}, reasons


def _priority(heatrisk_level, vulnerability_score):
    band = next(
        (
            item["id"]
            for item in SCORING["bands"]
            if item["min"] <= vulnerability_score <= item["max"]
        ),
        None,
    )
    if band is None:
        raise ValueError("vulnerability_score is outside configured bands")
    column = MATRIX_COLUMNS.index(band)
    return MATRIX[str(heatrisk_level)][column]


def _applicable_actions(priority, patient):
    if priority is None:
        return []

    actions = list(MODEL["actions"][priority])
    if priority == "routine":
        return actions

    if patient.get("has_reliable_home_cooling") is True:
        actions = [value for value in actions if value != "CONFIRM_COOLING_PLAN"]
    if (
        patient.get("expected_heat_exposure") is False
        and patient.get("has_strenuous_heat_exposure") is False
    ):
        actions = [value for value in actions if value != "REDUCE_HEAT_EXPOSURE"]
    if (
        patient.get("has_reliable_checkin_support") is True
        and patient.get("needs_assistance_for_heat_protection") is False
    ):
        actions = [value for value in actions if value != "ARRANGE_CHECKIN"]
    return actions


def _decision_relevant_missing_fields(patient):
    missing = [
        field for field in CLINICAL_BOOLEAN_FIELDS if patient.get(field) is None
    ]

    if patient.get("medication_classes") is None:
        missing.append("medication_classes")
    if patient.get("medication_review_complete") is not True:
        missing.append("medication_review_complete")

    home_cooling = patient.get("has_reliable_home_cooling")
    alternative_cooling = patient.get("can_access_cooling_location")
    if home_cooling is None:
        missing.append("has_reliable_home_cooling")
        if alternative_cooling is None:
            missing.append("can_access_cooling_location")
    elif home_cooling is False and alternative_cooling is None:
        missing.append("can_access_cooling_location")

    for field in [
        "has_reliable_checkin_support",
        "needs_assistance_for_heat_protection",
    ]:
        if patient.get(field) is None:
            missing.append(field)

    return list(dict.fromkeys(missing))


def assess_heat_outreach(patient, heatrisk_level, forecast_date):
    """Assess one adult with hypertension using four-domain rule v0.3.0.

    Missing keys and None remain unknown. HeatRisk must be 0 through 4 or None.
    The function returns a 0–4 clinical/medication subtotal plus the 0–8
    vulnerability score used by the outreach matrix. Heat exposure is not scored.
    """
    patient = dict(patient or {})
    personalization_warnings = []
    for field in EXPOSURE_ACTION_CONTEXT_FIELDS:
        value = patient.get(field)
        if value is not None and not isinstance(value, bool):
            personalization_warnings.append(
                f"{field} was ignored for action personalization; expected True, False, or None"
            )
            patient[field] = None
    if (
        patient.get("expected_heat_exposure") is False
        and patient.get("has_strenuous_heat_exposure") is True
    ):
        personalization_warnings.append(
            "Contradictory optional heat-exposure context was ignored for action personalization"
        )
        patient["expected_heat_exposure"] = None
        patient["has_strenuous_heat_exposure"] = None

    symptoms = patient.get("current_symptoms")
    symptom_pathway, immediate_message, symptom_reasons = _symptom_route(
        symptoms if isinstance(symptoms, list) or symptoms is None else None
    )

    result = {
        "assessment_id": str(uuid4()),
        "patient_id": patient.get("patient_id"),
        "rule_version": MODEL["version"],
        "assessment_time": datetime.now(timezone.utc).isoformat(),
        "forecast_date": forecast_date,
        "heatrisk_level": heatrisk_level,
        "score_name": SCORING["score_name"],
        "score_max": max(item["max"] for item in SCORING["bands"]),
        "domain_score_ranges": {},
        "clinical_medication_score": None,
        "clinical_medication_score_range": None,
        "vulnerability_score": None,
        "vulnerability_score_range": None,
        "outreach_priority": None,
        "possible_priorities": [],
        "assessment_status": None,
        "symptom_pathway": symptom_pathway,
        "immediate_message": immediate_message,
        "outreach_message": None,
        "safety_message": (
            "Do not change medications or prescribed fluid limits based on this tool."
        ),
        "reason_codes": list(symptom_reasons),
        "action_ids": [],
        "missing_fields": [],
        "personalization_warnings": personalization_warnings,
        "source_refs": ["E08"] if symptom_pathway != "none_reported" else [],
        "validation_status": "prototype_not_clinically_validated",
        "validation_errors": [],
    }

    errors = []
    patient_id = patient.get("patient_id")
    if not isinstance(patient_id, str) or not patient_id.strip():
        errors.append("patient_id must be a non-empty pseudonymous string")

    age = patient.get("age")
    if age is not None and (
        isinstance(age, bool)
        or not isinstance(age, int)
        or not 0 <= age <= 130
    ):
        errors.append("age must be an integer from 0 through 130, or None")
    for field in CORE_BOOLEAN_FIELDS:
        value = patient.get(field)
        if value is not None and not isinstance(value, bool):
            errors.append(f"{field} must be True, False, or None")

    medication_classes = patient.get("medication_classes")
    if medication_classes is not None and (
        not isinstance(medication_classes, list)
        or not all(isinstance(value, str) and value.strip() for value in medication_classes)
    ):
        errors.append("medication_classes must be a list of normalized class names, or None")
    if patient.get("medication_review_complete") is True and medication_classes is None:
        errors.append(
            "medication_classes cannot be None when medication_review_complete is True; "
            "use [] after a complete review confirms no mapped heat-relevant medicines"
        )

    if symptoms is not None and (
        not isinstance(symptoms, list)
        or not all(isinstance(value, str) and value.strip() for value in symptoms)
    ):
        errors.append("current_symptoms must be a list of symptom names, or None")

    if heatrisk_level is not None and (
        isinstance(heatrisk_level, bool)
        or not isinstance(heatrisk_level, int)
        or not 0 <= heatrisk_level <= 4
    ):
        errors.append("heatrisk_level must be an integer from 0 through 4, or None")

    try:
        if not isinstance(forecast_date, str):
            raise ValueError
        date.fromisoformat(forecast_date)
    except ValueError:
        errors.append("forecast_date must use YYYY-MM-DD format")

    if errors:
        result.update(
            assessment_status="invalid_input",
            validation_errors=errors,
            reason_codes=list(dict.fromkeys(symptom_reasons + ["INVALID_INPUT"])),
            outreach_message="Correct the input errors before calculating outreach.",
        )
        return result

    if (age is not None and age < 18) or patient.get("has_hypertension") is False:
        reasons = list(symptom_reasons)
        if age is not None and age < 18:
            reasons.append("PATIENT_UNDER_18")
        if patient.get("has_hypertension") is False:
            reasons.append("HYPERTENSION_NOT_CONFIRMED")
        result.update(
            assessment_status="out_of_scope",
            reason_codes=list(dict.fromkeys(reasons)),
            outreach_message="This model is only scoped to adults with confirmed hypertension.",
        )
        return result

    eligibility_missing = []
    if age is None:
        eligibility_missing.append("age")
    if patient.get("has_hypertension") is None:
        eligibility_missing.append("has_hypertension")
    if eligibility_missing:
        result.update(
            assessment_status="needs_information",
            missing_fields=eligibility_missing,
            reason_codes=list(
                dict.fromkeys(symptom_reasons + ["ELIGIBILITY_NOT_CONFIRMED"])
            ),
            outreach_message="Confirm adult age and hypertension before scoring.",
        )
        return result

    cooling, cooling_reasons = _cooling_score(patient)
    support, support_reasons = _support_score(patient)
    domains = {
        "clinical": _clinical_score(patient),
        "medication": _medication_score(patient),
        "cooling": cooling,
        "support": support,
    }
    medical_total = {
        "min": domains["clinical"]["min"] + domains["medication"]["min"],
        "max": domains["clinical"]["max"] + domains["medication"]["max"],
    }
    total = {
        "min": sum(value["min"] for value in domains.values()),
        "max": sum(value["max"] for value in domains.values()),
    }

    missing = _decision_relevant_missing_fields(patient)
    if symptoms is None:
        missing.append("current_symptoms")
    if patient.get("has_prescribed_fluid_restriction") is None:
        missing.append("has_prescribed_fluid_restriction")
    missing = list(dict.fromkeys(missing))

    exact_medical = (
        medical_total["min"]
        if medical_total["min"] == medical_total["max"]
        else None
    )
    exact_score = total["min"] if total["min"] == total["max"] else None
    result.update(
        domain_score_ranges=domains,
        clinical_medication_score=exact_medical,
        clinical_medication_score_range=medical_total,
        vulnerability_score=exact_score,
        vulnerability_score_range=total,
        missing_fields=missing,
        reason_codes=list(
            dict.fromkeys(result["reason_codes"] + cooling_reasons + support_reasons)
        ),
        source_refs=["E01", "E03", "E04", "E05", "E06", "E07", "E10"]
        + (["E08"] if symptom_pathway != "none_reported" else []),
    )

    if heatrisk_level is None:
        result.update(
            assessment_status="forecast_unavailable",
            reason_codes=list(
                dict.fromkeys(result["reason_codes"] + ["FORECAST_UNAVAILABLE"])
            ),
            action_ids=["MAINTAIN_HEAT_PLAN"],
            outreach_message=(
                "Official HeatRisk is unavailable. Keep the heat plan ready and "
                "reassess when a verified forecast is available."
            ),
        )
        return result

    possible = {
        _priority(heatrisk_level, score)
        for score in range(total["min"], total["max"] + 1)
    }
    possible_priorities = [value for value in PRIORITY_ORDER if value in possible]
    priority = possible_priorities[0] if len(possible_priorities) == 1 else None

    if priority is None:
        status = "needs_information"
        message = "More information is needed because the score range changes outreach priority."
        reasons = result["reason_codes"] + ["MISSING_INFORMATION_AFFECTS_PRIORITY"]
    else:
        status = "partial" if missing else "complete"
        messages = {
            "routine": "Routine outreach: maintain the existing heat plan.",
            "targeted": "Targeted outreach: provide personalized prevention and support checks.",
            "priority": (
                "Priority outreach: flag for care-team review before the forecast heat period. "
                "This is not an emergency diagnosis."
            ),
        }
        message = messages[priority]
        reasons = result["reason_codes"]

    result.update(
        assessment_status=status,
        outreach_priority=priority,
        possible_priorities=possible_priorities,
        outreach_message=message,
        reason_codes=list(dict.fromkeys(reasons + [f"HEATRISK_LEVEL_{heatrisk_level}"])),
        action_ids=_applicable_actions(priority, patient),
    )
    return result


def _format_score_range(score_range):
    if not score_range:
        return "not calculated"
    if score_range["min"] == score_range["max"]:
        return str(score_range["min"])
    return f'{score_range["min"]}–{score_range["max"]}'


def show_assessment(result):
    """Print the small set of fields needed for review."""
    print("Preventive assessment status:", result["assessment_status"])
    print("Symptom pathway:", result["symptom_pathway"])
    if result["immediate_message"]:
        print("Immediate message:", result["immediate_message"])

    print("HeatRisk (H):", result["heatrisk_level"])
    forecast = result.get("forecast_source")
    if forecast:
        print("Forecast location:", forecast["location_id"])
        print("Forecast date:", forecast["forecast_date"])
        if forecast["heatrisk_label"]:
            print("NWS HeatRisk label:", forecast["heatrisk_label"])
        if forecast["error"]:
            print("Forecast lookup note:", forecast["error"])

    medical_range = result.get("clinical_medication_score_range")
    if medical_range:
        print(
            "Clinical + medication subtotal (0–4):",
            _format_score_range(medical_range),
        )
    vulnerability_range = result.get("vulnerability_score_range")
    if vulnerability_range:
        print(
            "Outreach vulnerability (V, 0–8):",
            _format_score_range(vulnerability_range),
        )
        print("Scored domains:", result["domain_score_ranges"])
        print("Not scored: heat exposure, cooling breaks, strenuous activity")

    print("Outreach priority:", result["outreach_priority"])
    if result["possible_priorities"]:
        print("Possible priorities:", result["possible_priorities"])
    print("Message:", result["outreach_message"])
    print("Safety:", result["safety_message"])
    print("Actions:", result["action_ids"])
    if result["missing_fields"]:
        print("Information needed:", result["missing_fields"])
    if result["validation_errors"]:
        print("Input errors:", result["validation_errors"])
    if result["personalization_warnings"]:
        print("Optional exposure-context notes:", result["personalization_warnings"])


## Recognize common hypertension medications

`classify_hypertension_medications_for_heat(...)` recognizes the common generic cardiovascular medication names explicitly illustrated in the [CDC Heat and Medications guidance](https://www.cdc.gov/heat-health/hcp/clinical-guidance/heat-and-medications-guidance-for-clinicians.html). It converts them to the normalized classes used by this prototype. `apply_hypertension_medications(...)` copies those fields into a patient dictionary for the risk-assessment function.

The output uses **heat-relevant**, **higher-concern combination**, and **needs review** labels. These are screening labels, not validated rankings of how dangerous an individual medication is. The list is deliberately non-exhaustive; an unrecognized medication keeps the medication review incomplete instead of being treated as safe. Never stop or change a medication based on this output.


In [2]:
CDC_HEAT_MEDICATION_GUIDANCE_URL = (
    "https://www.cdc.gov/heat-health/hcp/clinical-guidance/"
    "heat-and-medications-guidance-for-clinicians.html"
)

# Generic examples named in the CDC cardiovascular-medication table.
# This is intentionally non-exhaustive and should be pharmacist-reviewed
# before any clinical deployment. Values are (canonical name, model class).
CDC_HYPERTENSION_MEDICATION_ALIASES = {
    "acetazolamide": ("acetazolamide", "DIURETIC"),
    "furosemide": ("furosemide", "DIURETIC"),
    "hydrochlorothiazide": ("hydrochlorothiazide", "DIURETIC"),
    "hctz": ("hydrochlorothiazide", "DIURETIC"),
    "atenolol": ("atenolol", "BETA_BLOCKER"),
    "metoprolol": ("metoprolol", "BETA_BLOCKER"),
    "propranolol": ("propranolol", "BETA_BLOCKER"),
    "amlodipine": ("amlodipine", "CALCIUM_CHANNEL_BLOCKER"),
    "felodipine": ("felodipine", "CALCIUM_CHANNEL_BLOCKER"),
    "nifedipine": ("nifedipine", "CALCIUM_CHANNEL_BLOCKER"),
    "enalapril": ("enalapril", "ACEI"),
    "lisinopril": ("lisinopril", "ACEI"),
    "ramipril": ("ramipril", "ACEI"),
    "losartan": ("losartan", "ARB"),
    "valsartan": ("valsartan", "ARB"),
    "glyceryl trinitrate": ("nitroglycerin", "NITRATE"),
    "nitroglycerin": ("nitroglycerin", "NITRATE"),
    "isosorbide mononitrate": ("isosorbide mononitrate", "NITRATE"),
}

HYPERTENSION_MEDICATION_CLASS_ORDER = [
    "DIURETIC",
    "ACEI",
    "ARB",
    "ARNI",
    "BETA_BLOCKER",
    "CALCIUM_CHANNEL_BLOCKER",
    "NITRATE",
]

CDC_CLASS_HEAT_CONCERNS = {
    "DIURETIC": [
        "volume depletion or dehydration",
        "electrolyte imbalance",
        "reduced thirst sensation",
    ],
    "ACEI": ["lower blood pressure or fainting/fall risk", "reduced thirst sensation"],
    "ARB": ["lower blood pressure or fainting/fall risk", "reduced thirst sensation"],
    "ARNI": ["CDC notes it may share the additive heat concern of an ARB"],
    "BETA_BLOCKER": [
        "reduced superficial vasodilation",
        "decreased sweating",
        "lower blood pressure or fainting/fall risk",
    ],
    "CALCIUM_CHANNEL_BLOCKER": [
        "lower blood pressure or fainting/fall risk",
        "electrolyte imbalance",
    ],
    "NITRATE": ["worsened hypotension"],
}


def _normalized_medication_text(value):
    return re.sub(r"[^a-z0-9]+", " ", value.strip().lower()).strip()


def _contains_medication_alias(text, alias):
    pattern = r"(?<![a-z0-9])" + r"\s+".join(
        re.escape(part) for part in alias.split()
    ) + r"(?![a-z0-9])"
    return re.search(pattern, text) is not None


def _split_medication_components(value):
    # Split explicit multi-ingredient separators while leaving dosage units
    # such as MG/ML intact. Sacubitril/valsartan is handled as a special pair.
    outer_parts = re.split(
        r"\s+(?:\+|\band\b)\s+", value.strip(), flags=re.IGNORECASE
    )
    parts = []
    for outer_part in outer_parts:
        normalized = _normalized_medication_text(outer_part)
        is_arni = (
            _contains_medication_alias(normalized, "sacubitril")
            and _contains_medication_alias(normalized, "valsartan")
        )
        if is_arni:
            parts.append(outer_part.strip())
        else:
            parts.extend(re.split(r"\s+/\s+", outer_part.strip()))
    return [part.strip() for part in parts if part.strip()]


def _recognize_hypertension_medication_entry(value):
    matched_names = []
    entry_classes = set()
    unmapped_components = []

    for component in _split_medication_components(value):
        normalized = _normalized_medication_text(component)
        component_matches = []
        component_classes = set()
        arni_match = (
            _contains_medication_alias(normalized, "sacubitril")
            and _contains_medication_alias(normalized, "valsartan")
        )
        if arni_match:
            component_matches.append("sacubitril/valsartan")
            component_classes.add("ARNI")

        for alias, (canonical_name, drug_class) in (
            CDC_HYPERTENSION_MEDICATION_ALIASES.items()
        ):
            if arni_match and canonical_name == "valsartan":
                continue
            if _contains_medication_alias(normalized, alias):
                component_matches.append(canonical_name)
                component_classes.add(drug_class)

        if not component_classes:
            class_token = component.strip().upper()
            if class_token in HYPERTENSION_MEDICATION_CLASS_ORDER:
                component_matches.append(class_token)
                component_classes.add(class_token)

        if component_classes:
            matched_names.extend(component_matches)
            entry_classes.update(component_classes)
        else:
            unmapped_components.append(component)

    return (
        list(dict.fromkeys(matched_names)),
        entry_classes,
        unmapped_components,
    )


def classify_hypertension_medications_for_heat(
    medications, medication_list_complete=None
):
    """Map common generic hypertension-drug names to prototype classes.

    Parameters
    ----------
    medications : str, list[str], or None
        Generic names or medication-display strings. Combination ingredients
        may be separated by a slash or plus sign. None means the list is unknown.
    medication_list_complete : bool or None
        Set True only when the supplied active medication list is complete.

    Returns
    -------
    dict
        Recognition details, model classes, score range, concern label, and
        patient fields that can be passed to the outreach assessment.
    """
    if medication_list_complete is not None and not isinstance(
        medication_list_complete, bool
    ):
        raise TypeError("medication_list_complete must be True, False, or None")

    if medications is None:
        raw_medications = []
        classes_value = None
    elif isinstance(medications, str):
        raw_medications = [medications]
        classes_value = []
    elif isinstance(medications, (list, tuple, set)):
        raw_medications = list(medications)
        classes_value = []
    else:
        raise TypeError("medications must be a string, a list of strings, or None")

    if any(not isinstance(item, str) or not item.strip() for item in raw_medications):
        raise ValueError("each medication must be a non-empty string")

    recognized = []
    unmapped = []
    all_classes = set()

    for original in raw_medications:
        entry_matches, entry_classes, unmapped_components = (
            _recognize_hypertension_medication_entry(original)
        )
        if entry_classes:
            all_classes.update(entry_classes)
            recognized.append(
                {
                    "input": original,
                    "matched_names": list(dict.fromkeys(entry_matches)),
                    "medication_classes": [
                        value
                        for value in HYPERTENSION_MEDICATION_CLASS_ORDER
                        if value in entry_classes
                    ],
                    "heat_relevance": "recognized_heat_relevant",
                }
            )
        unmapped.extend(unmapped_components)

    ordered_classes = [
        value for value in HYPERTENSION_MEDICATION_CLASS_ORDER if value in all_classes
    ]
    if classes_value is not None:
        classes_value = ordered_classes

    if medications is None:
        review_complete = None
    elif unmapped:
        review_complete = False
    else:
        review_complete = medication_list_complete
    patient_fields = {
        "medication_classes": classes_value,
        "medication_review_complete": review_complete,
    }
    score_range = _medication_score(patient_fields)
    higher_concern_combo = (
        "DIURETIC" in all_classes
        and bool(all_classes & {"ACEI", "ARB", "ARNI"})
    )

    if higher_concern_combo:
        concern = "higher_concern_combination"
        explanation = (
            "Diuretic plus ACE inhibitor, ARB, or ARNI: the prototype assigns "
            "the medication-domain maximum and routes through the outreach matrix."
        )
    elif ordered_classes and review_complete is True:
        concern = "heat_relevant_medication"
        explanation = "At least one CDC-listed heat-relevant medication class was recognized."
    elif ordered_classes:
        concern = "heat_relevant_needs_review"
        explanation = (
            "A heat-relevant class was recognized, but the medication list is not "
            "confirmed complete or contains an unmapped entry."
        )
    elif review_complete is True:
        concern = "no_recognized_heat_relevant_medication"
        explanation = (
            "No medication in this limited hypertension mapping was recognized as "
            "heat-relevant; this is not a declaration that the regimen is heat-safe."
        )
    else:
        concern = "needs_medication_review"
        explanation = (
            "Medication information is incomplete or unmapped, so the model keeps "
            "the medication score uncertain."
        )

    return {
        "model_heat_concern": concern,
        "explanation": explanation,
        "recognized_medications": recognized,
        "unmapped_medications": unmapped,
        "mapping_complete": medications is not None and not unmapped,
        "medication_list_complete": medication_list_complete,
        "medication_classes": classes_value,
        "medication_review_complete": review_complete,
        "model_medication_score_range": score_range,
        "class_heat_concerns": {
            value: CDC_CLASS_HEAT_CONCERNS[value] for value in ordered_classes
        },
        "patient_fields": patient_fields,
        "source_url": CDC_HEAT_MEDICATION_GUIDANCE_URL,
        "safety_message": (
            "Screening label only; not an individual danger rating. Do not stop or "
            "change medications or prescribed fluid limits based on this output."
        ),
    }


def apply_hypertension_medications(
    patient, medications, medication_list_complete=None
):
    """Return a copied patient record populated for the outreach scorer."""
    if patient is not None and not isinstance(patient, dict):
        raise TypeError("patient must be a dictionary or None")
    classification = classify_hypertension_medications_for_heat(
        medications=medications,
        medication_list_complete=medication_list_complete,
    )
    updated_patient = dict(patient or {})
    updated_patient.update(classification["patient_fields"])
    return updated_patient, classification


## Live NWS HeatRisk lookup

This nationwide helper calls the [NWS NDFD XML REST service](https://digital.weather.gov/xml/rest.php) with a five-digit U.S. ZIP code and requests the `heatrisk` element. NWS currently labels HeatRisk experimental. The response supplies local valid times and one of five categories, which are converted to `0–4`. When no date is supplied, the helper derives today's local date from the UTC offset in the ZIP's NWS forecast. The lookup verifies location, issue time, valid period, and a configurable three-hour freshness limit. If the ZIP is unknown to NWS or the feed, date, location, timestamp, or value is unusable, it keeps `heatrisk_level=None` and returns `forecast_unavailable` rather than treating it as zero. Urgent symptom text is emitted before the network request.


In [3]:
from datetime import time, timedelta
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import xml.etree.ElementTree as ET


NWS_NDFD_ENDPOINT = (
    "https://digital.weather.gov/xml/sample_products/"
    "browser_interface/ndfdXMLclient.php"
)
NWS_HEATRISK_LEVELS = {
    "little to none": 0,
    "minor": 1,
    "moderate": 2,
    "major": 3,
    "extreme": 4,
}


def get_nws_heatrisk_for_zip(
    zip_code, forecast_date=None, timeout=20, max_age_minutes=180
):
    """Retrieve NWS Experimental HeatRisk for one U.S. ZIP and local date.

    Parameters
    ----------
    zip_code : str or int
        Five-digit U.S. ZIP code. Use a string to preserve a leading zero.
    forecast_date : str, date, or None
        Local YYYY-MM-DD date. None selects today's date at the ZIP.
    timeout : int or float
        Network timeout in seconds.
    max_age_minutes : int, float, or None
        Maximum accepted age of the NWS response. None disables the age check.

    Returns
    -------
    dict
        Numeric HeatRisk plus source, location, issue, retrieval, and valid times.
    """
    zip_code = str(zip_code).strip()
    if not re.fullmatch(r"\d{5}", zip_code):
        raise ValueError("zip_code must contain exactly five digits")
    if max_age_minutes is not None and (
        isinstance(max_age_minutes, bool)
        or not isinstance(max_age_minutes, (int, float))
        or max_age_minutes <= 0
    ):
        raise ValueError("max_age_minutes must be positive or None")

    automatic_date = forecast_date is None
    if automatic_date:
        # Use UTC only to build a wide request window. After the response,
        # replace this with today's date in the forecast's local UTC offset.
        target_date = datetime.now(timezone.utc).date()
    elif isinstance(forecast_date, datetime):
        target_date = forecast_date.date()
    elif isinstance(forecast_date, date):
        target_date = forecast_date
    else:
        target_date = date.fromisoformat(str(forecast_date))

    result = {
        "location_id": f"ZIP:{zip_code}",
        "zip_code": zip_code,
        "local_timezone": None,
        "forecast_date": target_date.isoformat(),
        "forecast_issued_at": None,
        "retrieved_at": datetime.now(timezone.utc).isoformat(),
        "forecast_age_minutes": None,
        "freshness_policy_minutes": max_age_minutes,
        "heatrisk_level": None,
        "heatrisk_label": None,
        "valid_start": None,
        "valid_end": None,
        "latitude": None,
        "longitude": None,
        "source_url": NWS_NDFD_ENDPOINT,
        "source_product_version": None,
        "data_mode": "live",
        "is_usable": False,
        "status": "forecast_unavailable",
        "error": None,
    }

    params = {
        "zipCodeList": zip_code,
        "product": "time-series",
        "begin": f"{(target_date - timedelta(days=1) if automatic_date else target_date).isoformat()}T00:00:00",
        "end": f"{(target_date + timedelta(days=3 if automatic_date else 2)).isoformat()}T00:00:00",
        "heatrisk": "heatrisk",
    }
    request_url = f"{NWS_NDFD_ENDPOINT}?{urlencode(params)}"
    request = Request(
        request_url,
        headers={"User-Agent": "HeatOutreachPrototype/0.1"},
    )

    try:
        with urlopen(request, timeout=timeout) as response:
            root = ET.fromstring(response.read())
    except (HTTPError, URLError, TimeoutError, ET.ParseError, OSError) as exc:
        result["error"] = f"NWS request failed: {exc}"
        return result

    retrieved_time = datetime.now(timezone.utc)
    result["retrieved_at"] = retrieved_time.isoformat()
    issued_text = root.findtext("./head/product/creation-date")
    if not issued_text:
        result["error"] = "NWS response did not include a creation timestamp."
        return result
    try:
        issued_time = datetime.fromisoformat(issued_text.replace("Z", "+00:00"))
        if issued_time.tzinfo is None:
            raise ValueError("timestamp has no UTC offset")
    except (AttributeError, TypeError, ValueError) as exc:
        result["error"] = f"NWS returned an invalid creation timestamp: {exc}"
        return result

    age_minutes = (retrieved_time - issued_time).total_seconds() / 60
    result["forecast_issued_at"] = issued_time.isoformat()
    result["forecast_age_minutes"] = round(age_minutes, 2)
    if age_minutes < -5:
        result["error"] = "NWS creation timestamp is unexpectedly in the future."
        return result
    if max_age_minutes is not None and age_minutes > max_age_minutes:
        result["error"] = (
            f"NWS response is stale ({age_minutes:.0f} minutes old; "
            f"limit {max_age_minutes:g})."
        )
        return result

    point = root.find(".//data/location/point")
    if point is None:
        result["error"] = "NWS response did not include a forecast location."
        return result
    try:
        result["latitude"] = float(point.attrib["latitude"])
        result["longitude"] = float(point.attrib["longitude"])
    except (KeyError, TypeError, ValueError):
        result["error"] = "NWS returned an invalid forecast location."
        return result

    heatrisk = root.find(".//data/parameters/heat-risk")
    if heatrisk is None:
        result["error"] = "NWS response did not contain HeatRisk data."
        return result

    layout_key = heatrisk.attrib.get("time-layout")
    layout = next(
        (
            item
            for item in root.findall(".//data/time-layout")
            if item.findtext("layout-key") == layout_key
        ),
        None,
    )
    if layout is None:
        result["error"] = "NWS response did not include the HeatRisk time layout."
        return result

    start_nodes = layout.findall("start-valid-time")
    end_nodes = layout.findall("end-valid-time")
    value_nodes = heatrisk.findall("value")
    if not (len(start_nodes) == len(end_nodes) == len(value_nodes)):
        result["error"] = "NWS returned mismatched HeatRisk values and times."
        return result

    records = []
    for start_node, end_node, value_node in zip(
        start_nodes, end_nodes, value_nodes
    ):
        # NWS may publish a nil first interval while the feed is refreshing.
        if not start_node.text or not end_node.text or not value_node.text:
            continue
        try:
            start = datetime.fromisoformat(
                start_node.text.replace("Z", "+00:00")
            )
            end = datetime.fromisoformat(end_node.text.replace("Z", "+00:00"))
            if start.tzinfo is None or end.tzinfo is None:
                continue
        except (AttributeError, TypeError, ValueError):
            continue
        records.append((start, end, value_node.text.strip()))

    if not records:
        result["error"] = "NWS returned no complete HeatRisk intervals."
        return result

    if automatic_date:
        target_date = datetime.now(records[0][0].tzinfo).date()
        result["forecast_date"] = target_date.isoformat()

    candidates = []
    for start, end, label in records:
        local_noon = datetime.combine(target_date, time(12), tzinfo=start.tzinfo)
        if start <= local_noon < end:
            candidates.append((start, end, label))

    if not candidates:
        candidates = [item for item in records if item[0].date() == target_date]
    if not candidates:
        result["error"] = "Requested date is outside the available HeatRisk forecast."
        return result

    start, end, label = max(candidates, key=lambda item: item[1] - item[0])
    result["local_timezone"] = str(start.tzinfo)
    normalized_label = " ".join(label.lower().replace("-", " " ).split())
    level = NWS_HEATRISK_LEVELS.get(normalized_label)
    if level is None:
        result["error"] = f"NWS returned an unknown HeatRisk label: {label or 'missing'}"
        return result

    result.update(
        heatrisk_level=level,
        heatrisk_label=label,
        valid_start=start.isoformat(),
        valid_end=end.isoformat(),
        is_usable=True,
        status="ok",
    )
    return result


def assess_patient_for_zip(
    patient,
    zip_code,
    forecast_date=None,
    timeout=20,
    max_age_minutes=180,
    on_symptom_message=print,
):
    """Route symptoms, fetch U.S. ZIP HeatRisk, then run preventive rules.

    ``on_symptom_message`` is called before any network request for emergency
    or review symptoms. Set it to a UI callback, or to None to suppress printing.
    """
    patient = dict(patient or {})
    symptoms = patient.get("current_symptoms")
    symptom_pathway, immediate_message, _ = _symptom_route(
        symptoms if isinstance(symptoms, list) or symptoms is None else None
    )
    if (
        symptom_pathway in {"emergency", "symptom_review"}
        and immediate_message
        and on_symptom_message is not None
    ):
        if not callable(on_symptom_message):
            raise TypeError("on_symptom_message must be callable or None")
        on_symptom_message(f"{symptom_pathway.upper()}: {immediate_message}")

    zip_code = str(zip_code).strip()
    if not re.fullmatch(r"\d{5}", zip_code):
        raise ValueError("zip_code must contain exactly five digits")
    expected_location = f"ZIP:{zip_code}"
    patient_location = patient.get("location_id")
    if not isinstance(patient_location, str) or (
        patient_location.strip().upper() not in {zip_code, expected_location}
    ):
        raise ValueError(
            f"patient location_id must match {expected_location}"
        )
    patient["location_id"] = expected_location

    forecast = get_nws_heatrisk_for_zip(
        zip_code=zip_code,
        forecast_date=forecast_date,
        timeout=timeout,
        max_age_minutes=max_age_minutes,
    )
    assessment = assess_heat_outreach(
        patient=patient,
        heatrisk_level=forecast["heatrisk_level"],
        forecast_date=forecast["forecast_date"],
    )
    assessment["location_id"] = expected_location
    assessment["forecast_source"] = forecast
    return assessment


## Example

Edit the patient dictionary, generic medication names, and ZIP code below. Cooling access and support/assistance now affect the score and priority. Heat-exposure fields are intentionally omitted because they are not scored. `apply_hypertension_medications(...)` recognizes the CDC-listed examples and fills `medication_classes`. Set `medication_list_complete=True` only when the supplied active list is complete. The example makes a live NWS request, so it requires internet access.


In [4]:
patient = {
    "patient_id": "demo-patient-001",
    "location_id": "ZIP:72301",
    "age": 72,
    "has_hypertension": True,
    "has_heart_failure": False,
    "has_chronic_kidney_disease": False,
    "has_diabetes": True,
    "history_of_heat_illness": False,
    # Cooling: no reliable home cooling, but a confirmed usable alternative.
    "has_reliable_home_cooling": False,
    "can_access_cooling_location": True,
    # Support: no reliable check-in person; no assistance needed for the plan.
    "has_reliable_checkin_support": False,
    "needs_assistance_for_heat_protection": False,
    "has_prescribed_fluid_restriction": False,
    "current_symptoms": [],
}

patient, medication_check = apply_hypertension_medications(
    patient,
    [
        "lisinopril 10 MG Oral Tablet",
        "hydrochlorothiazide 25 MG Oral Tablet",
    ],
    medication_list_complete=True,
)
print("Medication heat check:", medication_check["model_heat_concern"])
print("Medication classes:", medication_check["medication_classes"])
print("Medication score range:", medication_check["model_medication_score_range"])

# Only the ZIP code is sent to NWS. None uses today's date at that ZIP.
zip_code = "72301"  # Arkansas
patient["location_id"] = f"ZIP:{zip_code}"
forecast_date = None

assessment = assess_patient_for_zip(patient, zip_code, forecast_date)
show_assessment(assessment)

# Offline/synthetic alternative:
# assessment = assess_heat_outreach(patient, heatrisk_level=2, forecast_date="2026-07-15")

# Machine-readable output:
# assessment


Medication heat check: higher_concern_combination
Medication classes: ['DIURETIC', 'ACEI']
Medication score range: {'min': 2, 'max': 2}
Preventive assessment status: complete
Symptom pathway: none_reported
HeatRisk (H): 3
Forecast location: ZIP:72301
Forecast date: 2026-09-20
NWS HeatRisk label: Major
Clinical + medication subtotal (0–4): 4
Outreach vulnerability (V, 0–8): 6
Scored domains: {'clinical': {'min': 2, 'max': 2}, 'medication': {'min': 2, 'max': 2}, 'cooling': {'min': 1, 'max': 1}, 'support': {'min': 1, 'max': 1}}
Not scored: heat exposure, cooling breaks, strenuous activity
Outreach priority: priority
Possible priorities: ['priority']
Message: Priority outreach: flag for care-team review before the forecast heat period. This is not an emergency diagnosis.
Safety: Do not change medications or prescribed fluid limits based on this tool.
Actions: ['CONFIRM_COOLING_PLAN', 'REDUCE_HEAT_EXPOSURE', 'ARRANGE_CHECKIN', 'REVIEW_EXISTING_HEAT_PLAN', 'CARE_TEAM_REVIEW']


In [5]:
assessment

{'assessment_id': 'b6bccc53-5738-4772-95de-250d2d0a1bf8',
 'patient_id': 'demo-patient-001',
 'rule_version': '0.3.0',
 'assessment_time': '2026-09-20T14:29:35.964881+00:00',
 'forecast_date': '2026-09-20',
 'heatrisk_level': 3,
 'score_name': 'outreach_vulnerability',
 'score_max': 8,
 'domain_score_ranges': {'clinical': {'min': 2, 'max': 2},
  'medication': {'min': 2, 'max': 2},
  'cooling': {'min': 1, 'max': 1},
  'support': {'min': 1, 'max': 1}},
 'clinical_medication_score': 4,
 'clinical_medication_score_range': {'min': 4, 'max': 4},
 'vulnerability_score': 6,
 'vulnerability_score_range': {'min': 6, 'max': 6},
 'outreach_priority': 'priority',
 'possible_priorities': ['priority'],
 'assessment_status': 'complete',
 'symptom_pathway': 'none_reported',
 'immediate_message': None,
 'outreach_message': 'Priority outreach: flag for care-team review before the forecast heat period. This is not an emergency diagnosis.',
 'safety_message': 'Do not change medications or prescribed fluid 

In [6]:
# Heat-exposure answers are optional action context only; they do not change score or priority.
# heatwave: long days of heat => tolerance goes down